# Clustering and Annotation

This notebook demonstrates cell clustering and cell type annotation workflows.

In [ ]:
import scanpy as sc
import scLucid
from scLucid.analysis import (
    cluster_cells,
    find_resolution,
    annotate_clusters,
    ClusteringConfig
)
from scLucid.utils.manager import get_marker_manager

## Load Preprocessed Data

In [ ]:
# Load preprocessed data
adata = sc.read_h5ad("results/preprocessed.h5ad")

print(f"Cells: {adata.n_obs}")
print(f"Genes: {adata.n_vars}")
print(f"PCs: {adata.obsm['X_pca'].shape[1]}")

## Optimize Clustering Resolution

In [ ]:
from scLucid.analysis import ResolutionSearchConfig

search_config = ResolutionSearchConfig(
    method="leiden",
    resolution_range=(0.2, 2.0, 10),
    compute_silhouette=True
)

optimal_res = find_resolution(adata, config=search_config)
print(f"Optimal resolution: {optimal_res}")

## Cluster Cells

In [ ]:
# Use optimal resolution or default
cluster_config = ClusteringConfig(
    method="leiden",
    resolution=1.0,  # or use optimal_res
    use_rep="X_pca",
    key_added="leiden"
)

adata = cluster_cells(adata, config=cluster_config)

# Visualize clusters
sc.pl.umap(adata, color=["leiden"], save="_clusters.pdf")

## Annotate Cell Types

In [ ]:
# Load marker database
mgr = get_marker_manager(species="human", tissue="Blood")
mgr.show_tree()

# Annotate clusters
adata = annotate_clusters(
    adata,
    marker_manager=mgr,
    method="markers",  # or "celltypist", "ensemble"
    min_confidence=0.5
)

# Visualize annotations
sc.pl.umap(adata, color=["leiden", "cell_type"], save="_annotation.pdf")

## Find Marker Genes

In [ ]:
from scLucid.analysis import find_markers, DifferentialConfig

de_config = DifferentialConfig(
    method="wilcoxon",
    groupby="cell_type"
)

markers = find_markers(adata, config=de_config)

# View top markers
sc.pl.rank_genes_groups_heatmap(
    adata,
    groupby="cell_type",
    n_genes=5,
    save="_markers.pdf"
)

## Save Results

In [ ]:
adata.write("results/annotated.h5ad")
print("✅ Analysis complete!")